In [4]:
import numpy as np
import pandas as pd
import os
from glob import glob
import json

In [293]:
arr = []
files = glob(os.path.join('json', '*.json'))

for file in files:
  with open (file, 'r') as f:
    try:
      contents = json.load(f)
      arr.extend(contents)
    except Exception as e:
      print(e)

# TODO: gist this (select key as k from table... translated to py)
select = [('ts','ts'), ('ms_played', 'ms_played'), ('reason_start', 'reason_start'), 
  ('reason_end', 'reason_end'), 
  ('master_metadata_track_name', 'track'), ('master_metadata_album_artist_name', 'artist'),
  ('master_metadata_album_album_name', 'album'), ('spotify_track_uri','uri')]
# data = np.array([{k[1]: i[k[0]] for k in select} for i in arr])

data = np.array([[i[k[0]] for k in select] for i in arr])
df = pd.DataFrame(data)
df.columns = [k[1] for k in select]
streams = df[df['ms_played'] >= 30000].copy()
streams['ts'] = pd.to_datetime(streams['ts']).dt.tz_convert('America/Vancouver')


streams = streams.set_index('ts').sort_index()
streams['date'] = streams.index.strftime('%Y-%m-%d')

In [278]:
monthly_top_artists = pd.DataFrame([])
months = pd.period_range(start="2018-01", end=streams.index[-1].strftime('%Y-%m'), freq='M')

for m in months:
  pivot_table = streams.loc[m.strftime('%Y-%m')].pivot_table(
      index='artist',
      values=['ms_played'],
      aggfunc={'ms_played':'sum'},
  ).sort_values(by='ms_played',ascending=False).head(50)
  monthly_top_artists[m] = pd.Series(pivot_table.index)

monthly_top_artists.index = list(range(1,51))
monthly_top_artists.to_csv('csv/monthly_top_artists.csv')


top_artists_pos = {}
for m, c in monthly_top_artists.iteritems():
  for index, artist in c.items():
    if artist not in top_artists_pos:
      top_artists_pos[artist] = {}
    top_artists_pos[artist][m.strftime('%Y-%m')] = index

top_artists_pos = pd.DataFrame.from_dict(top_artists_pos, orient='index').filter(like="2025").loc[(test <= 10).any(axis=1)]
top_artists_pos.to_csv('csv/top_artists_pos.csv')

In [211]:
# should have dated_streams = streams.loc[x] higher up here

top_artists = streams.loc["2025"].pivot_table(
    index='artist',
    values=['ms_played', 'uri', 'track', 'date'],
    aggfunc={'ms_played':'sum', 'uri': 'count', 'track': 'nunique', 'date': 'nunique'},
).sort_values(by='ms_played',ascending=False).head(50)
top_artists['h'] = round(top_artists['ms_played'] / 1000 / 60 / 60, 1)
top_artists.to_csv('csv/top_artists.csv')

top_albums = streams.loc["2025"].pivot_table(
    index='album',
    values=['ms_played', 'uri', 'date'],
    aggfunc={'ms_played':'sum', 'uri': 'count', 'date': 'nunique'},
).sort_values(by='ms_played',ascending=False).head(50)
top_albums['h'] = round(top_albums['ms_played'] / 1000 / 60 / 60, 1)
top_albums.to_csv('csv/top_albums.csv')

top_tracks = streams.loc["2025"].pivot_table(
    index='track',
    values=['ms_played', 'uri', 'date'],
    aggfunc={'ms_played':'sum', 'uri': 'count', 'date': 'nunique'},
).sort_values(by='ms_played',ascending=False).head(50)
top_tracks['h'] = round(top_tracks['ms_played'] / 1000 / 60 / 60, 1)
top_tracks.to_csv('csv/top_tracks.csv')

In [230]:
skips = df[(df['ms_played'] < 30000) & (df['reason_end'] == "fwdbtn")].copy()
skips['ts'] = pd.to_datetime(skips['ts'])
skips = skips.set_index('ts').sort_index()
skips['date'] = skips.index.strftime('%Y-%m-%d')

top_skips = skips.loc['2025'].pivot_table(
    index='track',
    values=['ms_played', 'uri', 'date'],
    aggfunc={'ms_played':'sum', 'uri': 'count', 'date': 'nunique'},
).sort_values(by='uri',ascending=False).head(50)

# top_skips

In [245]:
streams.agg({"ms_played": "sum", "uri": "count", "artist": "nunique", "track": "nunique"})

df["ms_played"].sum()

11454297633

In [316]:
day_matrix = [{} for i in range(7)]
for d in range(7):
  for t in ["early morning", "morning", "midday", "afternoon", "evening", "night", "late night"]:
    day_matrix[d][t] = 0

# ["early morning", "morning", "midday", "afternoon", "evening", "night", "late night"]
# 12-5, 5-10, 10-1, 1-4, 4-7, 7-10, 10-12

dated_streams = streams.loc["2025"]
for ts, ms_played in zip(dated_streams.index, dated_streams["ms_played"]):
  tod = "late night"
  if ts.hour < 6: tod = "early morning"
  elif ts.hour < 10: tod = "morning"
  elif ts.hour < 13: tod = "midday"
  elif ts.hour < 16: tod = "afternoon"
  elif ts.hour < 19: tod = "evening"
  elif ts.hour < 22: tod = "night"
  day_matrix[ts.day_of_week][tod] += ms_played

day_matrix = pd.DataFrame(day_matrix)
day_matrix.index = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
day_matrix = day_matrix.T / 1000 / 60 / 60
day_matrix.insert(loc=0, column='Range', value=["12-6", "6-10", "10-1", "1-4", "4-7", "7-10", "10-12"])

day_matrix

,Range,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday
early morning,12-6,0.017035,0.209309,0.000000,0.024118,0.072298,0.094171,0.199740
morning,6-10,12.807221,11.096722,10.643578,10.509560,10.288081,1.879979,1.329129
midday,10-1,7.013744,0.459056,3.670384,5.378767,5.306465,5.273129,5.052710
afternoon,1-4,21.185134,14.042199,16.246407,15.794789,20.474146,9.311332,16.670606
evening,4-7,10.434719,10.207762,14.375376,9.598516,10.426149,7.252989,11.365964
night,7-10,7.126437,4.629164,8.656528,7.140731,4.492882,3.453877,7.507265
late night,10-12,0.205176,0.151321,1.064565,0.451992,0.920731,0.764358,1.974611
